# Notebook 3 — LSTM Surge Detection
**SmartRetail: Predictive Demand Forecasting & Dynamic Pricing Engine**

### What this notebook does
1. Loads `train/` and `test/` Parquet **independently** from notebook 1's output — this
   notebook does not reuse notebook 2's in-memory features
2. Labels each SKU-day as a **surge** if `daily_views` exceeds that item's train-window
   mean + 2 standard deviations — stats fit on train only, applied to both splits
3. Selects the 5-feature sequence input (`daily_views`, `daily_addtocarts`,
   `velocity_7d`, `day_of_week`, `base_price`) and min-max normalises it, again fit on
   train only
4. Collects the flat (non-windowed) feature table to pandas — small and cheap, unlike
   collecting pre-built sequences
5. Builds 7-day sliding windows per item in NumPy (`sliding_window_view`), independently
   within each split
6. Trains a binary-classification LSTM (`SurgeLSTM`) with a `pos_weight`-weighted loss to
   counter the fact that surges are rare by construction
7. Evaluates on the held-out test slice (accuracy, precision, recall, F1, confusion matrix)
8. Writes `models/lstm_model.pth` (weights) and `models/lstm_config.json` (feature order,
   normalisation stats, architecture, decision threshold) — Phase 2's `consumer_engine.py`
   needs both to reconstruct the model and normalise live stream data identically

---
### Before you run
Uses the same Drive layout as notebooks 1 & 2:
```
MyDrive/
└── SmartRetail/
    ├── data/
    │   └── processed/
    │       ├── train/
    │       └── test/
    └── models/
        ├── lstm_model.pth     ← written by this notebook
        └── lstm_config.json   ← written by this notebook
```


## Step 1 — Install PySpark

In [ ]:
!pip install pyspark==3.5.1 -q
print("PySpark installed.")

## Step 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")

## Step 3 — Configuration
**Only edit this cell.** All paths and hyperparameters below derive from these.

In [ ]:
# ── Edit these two lines to match your Drive layout ────────────────────────────
DATA_PATH   = "/content/drive/MyDrive/SmartRetail/data/processed/"
MODELS_PATH = "/content/drive/MyDrive/SmartRetail/models/"
# ───────────────────────────────────────────────────────────────────────────────

TRAIN_PARQUET    = DATA_PATH + "train/"
TEST_PARQUET     = DATA_PATH + "test/"
LSTM_MODEL_PATH  = MODELS_PATH + "lstm_model.pth"
LSTM_CONFIG_JSON = MODELS_PATH + "lstm_config.json"

WINDOW_SIZE           = 7      # 7 days of history per sequence
SURGE_STD_MULTIPLIER  = 2.0    # surge = views > mean + 2*std
SEQUENCE_FEATURES     = ["daily_views", "daily_addtocarts", "velocity_7d", "day_of_week", "base_price"]
DECISION_THRESHOLD    = 0.5

print("Config ready.")
print(f"  Train in     : {TRAIN_PARQUET}")
print(f"  Test in      : {TEST_PARQUET}")
print(f"  Model out    : {LSTM_MODEL_PATH}")
print(f"  Config out   : {LSTM_CONFIG_JSON}")
print(f"  Window size  : {WINDOW_SIZE}")
print(f"  Features     : {SEQUENCE_FEATURES}")

## Step 4 — Initialise SparkSession

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("SmartRetail-LSTM")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.shuffle.partitions", "100")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")
print(f"Spark UI      : {spark.sparkContext.uiWebUrl}")

## Step 5 — Load Train / Test Parquet
Loaded fresh from notebook 1's output — `daily_views` and the other raw columns are
still present here, unlike notebook 2's copies which dropped them to avoid leakage on
a *different* target.

In [ ]:
from pyspark.sql import functions as F

train_df = spark.read.parquet(TRAIN_PARQUET)
test_df  = spark.read.parquet(TEST_PARQUET)

print(f"Train rows : {train_df.count():,}")
print(f"Test rows  : {test_df.count():,}")
train_df.printSchema()

## Step 6 — Engineer the Surge Label
A day is a **surge** if `daily_views` exceeds that item's own historical mean by more
than `SURGE_STD_MULTIPLIER` standard deviations. Stats are computed on `train_df` only
and reused on `test_df` — the same leak-safety rule as notebook 2's `StringIndexer`.
Items with a single training row get `std_views = 0` (via `coalesce`), which makes the
threshold equal to the mean — never satisfied by a value that equals it, so it can't
manufacture false positives.

Items absent from `train_df` have no baseline to compare against and are dropped via
inner join — mirrors notebook 2's `items_with_sales` filter.

Surges are rare *by construction* (a >2σ threshold), and the test window loses more
than train to windowing later — so we print the class balance now, before investing in
a training run that a near-empty positive class would make meaningless.

In [ ]:
item_view_stats = (
    train_df.groupBy("itemid")
    .agg(
        F.mean("daily_views").alias("mean_views"),
        F.coalesce(F.stddev("daily_views"), F.lit(0.0)).alias("std_views"),
    )
)

def label_surge(df):
    return (
        df.join(item_view_stats, on="itemid", how="inner")
        .withColumn(
            "is_surge",
            F.when(
                F.col("daily_views") > (F.col("mean_views") + SURGE_STD_MULTIPLIER * F.col("std_views")),
                F.lit(1)
            ).otherwise(F.lit(0))
        )
    )

train_labeled = label_surge(train_df).cache()
test_labeled  = label_surge(test_df).cache()

def report_surge_rate(df, name):
    total = df.count()
    positives = df.filter(F.col("is_surge") == 1).count()
    print(f"{name:5s}: {positives:,} / {total:,} surge rows ({positives / total * 100:.2f}%)")

report_surge_rate(train_labeled, "Train")
report_surge_rate(test_labeled, "Test")

## Step 7 — Select & Normalise Sequence Features
Only `daily_views`, `daily_addtocarts`, `velocity_7d`, `day_of_week`, `base_price` go
into the LSTM input — kept tight to avoid noise. Min/max are fit on `train_labeled`
only and reused on `test_labeled` (never refit on test). A zero-range feature (min ==
max) is mapped to a constant `0.0` instead of dividing by zero.

In [ ]:
feature_stats = train_labeled.select(
    *[F.min(c).alias(f"{c}_min") for c in SEQUENCE_FEATURES],
    *[F.max(c).alias(f"{c}_max") for c in SEQUENCE_FEATURES],
).collect()[0].asDict()

print("Feature min/max (fit on train only):")
for c in SEQUENCE_FEATURES:
    print(f"  {c:15s} min={feature_stats[f'{c}_min']:.4f}  max={feature_stats[f'{c}_max']:.4f}")

def normalize(df):
    out = df
    for c in SEQUENCE_FEATURES:
        col_min = feature_stats[f"{c}_min"]
        col_max = feature_stats[f"{c}_max"]
        col_range = col_max - col_min
        if col_range == 0:
            out = out.withColumn(f"{c}_norm", F.lit(0.0))
        else:
            out = out.withColumn(
                f"{c}_norm",
                F.least(F.lit(1.0), F.greatest(F.lit(0.0), (F.col(c) - F.lit(col_min)) / F.lit(col_range)))
            )
    return out

train_norm = normalize(train_labeled)
test_norm  = normalize(test_labeled)

NORM_FEATURE_COLS = [f"{c}_norm" for c in SEQUENCE_FEATURES]
train_norm.select("itemid", "event_date", *NORM_FEATURE_COLS, "is_surge").show(5)

## Step 8 — Collect Flat Data to Pandas
We collect the **flat** per-day table (5 normalised features + label, one row per
SKU-day) rather than pre-building sequences in Spark — a 7-wide `collect_list` window
would ship roughly 7× as much data across the Spark→driver boundary for no benefit,
since the windows still have to be reshaped in NumPy either way. The flat table is
small, so this collect is cheap.

Spark's job ends here — everything from this point on is plain pandas / NumPy /
PyTorch, so we free the cluster now rather than holding it idle through training.

In [ ]:
SELECT_COLS = ["itemid", "event_date", *NORM_FEATURE_COLS, "is_surge"]

train_flat = (
    train_norm.select(*SELECT_COLS)
    .orderBy("itemid", "event_date")
    .toPandas()
)
test_flat = (
    test_norm.select(*SELECT_COLS)
    .orderBy("itemid", "event_date")
    .toPandas()
)

print(f"Train flat rows: {len(train_flat):,}")
print(f"Test flat rows : {len(test_flat):,}")

spark.stop()
print("SparkSession closed — remaining steps run in plain pandas / NumPy / PyTorch.")

## Step 9 — Build Sliding Windows (NumPy)
For each item, slide a `WINDOW_SIZE`-day window across its sorted rows using
`sliding_window_view` — one call reshapes an item's whole history into all its windows
at once, no per-window Python loop. The label for each window is `is_surge` on the
**last** day of that window, which is simply the label already sitting on the row the
window ends at.

Windows are built **independently within train and within test** — no bridging across
the split boundary. That means the first `WINDOW_SIZE - 1` rows of each item's test
history don't get a full window and are dropped. This is a small, accepted loss that
keeps the train/test boundary strict, consistent with how notebook 2 handled its own
lag-feature warm-up rows.

In [ ]:
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view

def build_windows(flat_df, feature_cols, window_size=WINDOW_SIZE):
    X_parts, y_parts = [], []

    for itemid, group in flat_df.sort_values(["itemid", "event_date"]).groupby("itemid"):
        if len(group) < window_size:
            continue
        features = group[feature_cols].to_numpy(dtype=np.float32)
        labels   = group["is_surge"].to_numpy(dtype=np.float32)

        # (T, F) -> (T - window_size + 1, F, window_size) -> (n_windows, window_size, F)
        windows = sliding_window_view(features, window_size, axis=0).transpose(0, 2, 1)
        window_labels = labels[window_size - 1:]   # label = is_surge of the LAST day in the window

        X_parts.append(windows)
        y_parts.append(window_labels)

    X = np.concatenate(X_parts, axis=0)
    y = np.concatenate(y_parts, axis=0)
    return X, y

X_train, y_train = build_windows(train_flat, NORM_FEATURE_COLS)
X_test,  y_test  = build_windows(test_flat, NORM_FEATURE_COLS)

print(f"X_train shape: {X_train.shape}   y_train shape: {y_train.shape}")
print(f"X_test  shape: {X_test.shape}   y_test  shape: {y_test.shape}")
print(f"Train surge rate (windowed): {y_train.mean() * 100:.2f}%")
print(f"Test  surge rate (windowed): {y_test.mean() * 100:.2f}%")

## Step 10 — Convert to PyTorch Tensors

In [ ]:
import torch

torch.manual_seed(42)   # covers weight init (Step 11) and DataLoader shuffling (Step 13)

X_train_t = torch.from_numpy(X_train)
y_train_t = torch.from_numpy(y_train)
X_test_t  = torch.from_numpy(X_test)
y_test_t  = torch.from_numpy(y_test)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"X_train_t: {tuple(X_train_t.shape)} {X_train_t.dtype}")
print(f"y_train_t: {tuple(y_train_t.shape)} {y_train_t.dtype}")

## Step 11 — Define the LSTM Model
A small binary classifier: an `LSTM` encodes the 7-day window, and a linear layer maps
its final hidden state to a single logit — surge vs. not-surge.

In [ ]:
import torch.nn as nn

class SurgeLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=32, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size,
                             num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        last_hidden = h_n[-1]                       # final layer's hidden state: (batch, hidden_size)
        return self.fc(last_hidden).squeeze(-1)      # raw logits: (batch,)

HIDDEN_SIZE = 32
NUM_LAYERS  = 1
INPUT_SIZE  = len(SEQUENCE_FEATURES)

model = SurgeLSTM(input_size=INPUT_SIZE, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS).to(device)
print(model)

## Step 12 — Handle Class Imbalance
Same lesson as notebook 2's zero-inflated `daily_transactions`: surges are rare by
construction, so a naive loss lets the model minimize error by always predicting
"no surge." We compute a `pos_weight` from the train class ratio and pass it straight
into `BCEWithLogitsLoss`, so the loss itself penalizes missed surges instead of hoping
the model discovers the imbalance on its own.

In [ ]:
n_pos = float(y_train.sum())
n_neg = float(len(y_train) - n_pos)
pos_weight_value = n_neg / max(n_pos, 1.0)
pos_weight = torch.tensor([pos_weight_value], device=device)

print(f"Train positives: {int(n_pos):,}  negatives: {int(n_neg):,}")
print(f"pos_weight for BCEWithLogitsLoss: {pos_weight_value:.2f}")

## Step 13 — Train the LSTM

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

BATCH_SIZE    = 256
EPOCHS        = 15
LEARNING_RATE = 1e-3

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

model.train()
for epoch in range(1, EPOCHS + 1):
    epoch_loss = 0.0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        optimizer.zero_grad()
        logits = model(batch_X)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * batch_X.size(0)

    epoch_loss /= len(train_loader.dataset)
    print(f"Epoch {epoch:2d}/{EPOCHS}  loss={epoch_loss:.4f}")

print("Training complete.")

## Step 14 — Evaluate on the Held-Out Test Slice
The label is a threshold on `daily_views`, which is itself one of the 5 sequence
inputs — expect the model to lean heavily on the current day's view count, with the
rest of the 7-day sequence adding modest lift on top. That's inherent to how the label
is defined, the same way `velocity_7d` dominated notebook 2's feature importances, not
a sign anything is broken.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

model.eval()
with torch.no_grad():
    test_logits = model(X_test_t.to(device))
    test_probs  = torch.sigmoid(test_logits).cpu().numpy()

test_preds = (test_probs >= DECISION_THRESHOLD).astype(int)
y_test_int = y_test.astype(int)

test_metrics = {
    "accuracy":  accuracy_score(y_test_int, test_preds),
    "precision": precision_score(y_test_int, test_preds, zero_division=0),
    "recall":    recall_score(y_test_int, test_preds, zero_division=0),
    "f1":        f1_score(y_test_int, test_preds, zero_division=0),
}

print("── Test set metrics ────────────────────────────")
for name, value in test_metrics.items():
    print(f"  {name:10s}: {value:.4f}")

print("\nConfusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_test_int, test_preds))

print(f"\nTest surge rate (windowed): {y_test_int.mean() * 100:.2f}%")

## Step 15 — Save Model & Config
Phase 2's `consumer_engine.py` has no Spark and never re-derives the min/max stats or
the surge-label rule — `lstm_config.json` is the only place they're recorded, so it has
to carry everything needed to reconstruct inference exactly: feature order, per-feature
normalisation range, architecture, and the decision threshold.

In [ ]:
import os
import json

os.makedirs(MODELS_PATH, exist_ok=True)

torch.save(model.state_dict(), LSTM_MODEL_PATH)
print(f"Model weights written to: {LSTM_MODEL_PATH}")

lstm_config = {
    "model_type": "SurgeLSTM",
    "window_size": WINDOW_SIZE,
    "sequence_features": SEQUENCE_FEATURES,   # order matches the tensor's feature axis
    "feature_min_max": {
        c: {"min": feature_stats[f"{c}_min"], "max": feature_stats[f"{c}_max"]}
        for c in SEQUENCE_FEATURES
    },
    "hyperparameters": {
        "input_size": INPUT_SIZE,
        "hidden_size": HIDDEN_SIZE,
        "num_layers": NUM_LAYERS,
    },
    "surge_label_rule": {
        "std_multiplier": SURGE_STD_MULTIPLIER,
        "description": "is_surge=1 if daily_views > item's train-window mean + std_multiplier * std",
    },
    "decision_threshold": DECISION_THRESHOLD,
    "test_metrics": {k: round(v, 4) for k, v in test_metrics.items()},
    "train_windows": int(len(y_train)),
    "test_windows": int(len(y_test)),
}

with open(LSTM_CONFIG_JSON, "w") as f:
    json.dump(lstm_config, f, indent=2)

print(f"Config written to: {LSTM_CONFIG_JSON}")
print(json.dumps(lstm_config, indent=2))

## Step 16 — Final Summary

In [ ]:
print("═" * 55)
print("  NOTEBOOK 3 — LSTM SURGE DETECTION TRAINING COMPLETE")
print("═" * 55)
print()
print(f"Test Accuracy  : {test_metrics['accuracy']:.4f}")
print(f"Test Precision : {test_metrics['precision']:.4f}")
print(f"Test Recall    : {test_metrics['recall']:.4f}")
print(f"Test F1        : {test_metrics['f1']:.4f}")
print()
print(f"Model weights → {LSTM_MODEL_PATH}")
print(f"Config        → {LSTM_CONFIG_JSON}")
print()
print("What Phase 2 (consumer_engine.py) expects:")
print("  Load lstm_config.json for feature order, min/max stats, and architecture.")
print("  Rebuild SurgeLSTM(input_size, hidden_size, num_layers) and load lstm_model.pth weights.")
print("  Normalise each incoming feature with the stored min/max before building a window.")
print("  Apply the stored decision_threshold to sigmoid(logits) to flag a surge.")